# Fase 2: Selección de Features, Validación y el Desafío del "BCI Illiteracy"

In [1]:
# --- 1. Reconstrucción del entorno
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PATH_REPO = '/content/drive/MyDrive/BCI-MENTORIA-UNC'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

# --- 2. Carga del checkpoint consolidado ---
ruta_checkpoint = os.path.join(PATH_REPO, "data_procesada", "features_largo_todos_sujetos.parquet")
df_check = pd.read_parquet(ruta_checkpoint)

print(df_check.shape)
print("Sujetos:", df_check['subject_id'].nunique())
print("Features distintas:", df_check['feature_col'].nunique())
print("NaN totales:", df_check.isna().sum().sum())

# --- 3. Pivot a formato ancho (una fila por trial, una columna por feature) ---
df_features_wide = df_check.pivot_table(
    index=['subject_id', 'trial_idx', 'label'],
    columns='feature_col',
    values='valor'
).reset_index()

df_features_wide['Clase_Nombre'] = df_features_wide['label'].map({0: 'Mano Izquierda', 1: 'Mano Derecha'})

print(f"\ndf_features_wide: {df_features_wide.shape}")
df_features_wide.head()

Mounted at /content/drive
(1427400, 5)
Sujetos: 60
Features distintas: 75
NaN totales: 0

df_features_wide: (19032, 79)


feature_col,subject_id,trial_idx,label,AR_coef10_C3,AR_coef10_C4,AR_coef10_Cz,AR_coef1_C3,AR_coef1_C4,AR_coef1_Cz,AR_coef2_C3,...,power_band_5_C3,power_band_5_C4,power_band_5_Cz,power_band_6_C3,power_band_6_C4,power_band_6_Cz,skewness_C3,skewness_C4,skewness_Cz,Clase_Nombre
0,1,0,0,0.081058,0.187386,0.212568,1.695716,1.903604,1.858320,-1.531086,...,0.012777,0.012250,0.008134,0.004992,0.007473,0.004192,0.147925,0.120641,0.092179,Mano Izquierda
1,1,1,0,0.096951,0.215023,0.086222,1.557386,1.953936,1.531764,-1.351885,...,0.008335,0.004211,0.004051,0.004453,0.005065,0.004099,0.160135,-0.416434,-0.231732,Mano Izquierda
2,1,2,1,0.057892,-0.009870,0.086022,1.782833,1.583895,1.862816,-1.615969,...,0.003964,0.002737,0.001527,0.003052,0.003420,0.001655,-0.428293,-0.668394,-1.041656,Mano Derecha
3,1,3,0,0.012777,0.015300,0.026840,1.513789,1.678217,1.496940,-1.271750,...,0.004383,0.004284,0.003306,0.007516,0.006560,0.004384,-0.077877,-0.207840,0.178673,Mano Izquierda
4,1,4,1,0.085348,0.215982,0.064743,1.714882,2.049973,2.155214,-1.484770,...,0.004604,0.003684,0.001643,0.001225,0.001046,0.000386,-1.150805,-1.093982,-1.305038,Mano Derecha


## Selección de features

Se selecciona inicialmente el top 20 de features más representativas, obtenidas de manera independiente por cada método de selección. Esta práctica asegura mayor solidez, ya que permite ejecutar los cuatro métodos con un valor de k amplio y, posteriormente, definir el subconjunto final de variables mediante un criterio de consenso. De este modo, el tamaño del conjunto definitivo no depende de una elección arbitraria previa, sino de la evidencia combinada aportada por los distintos enfoques.

- Selección de Features Dentro de la Validación

En vez de elegir "las mejores features" una sola vez usando todos los datos, se hace por partes: para cada fold de validación, se recalcula la selección de features usando solo los datos de entrenamiento de ese fold, sin tocar los datos de prueba.

¿Por qué? Si se selecciona las features mirando el 100% de los datos, y después se usan esos mismos datos para medir qué tan bien funciona el modelo, se está haciendo trampa sin querer: el modelo ya tuvo "pistas". Eso hace que el resultado parezca mejor de lo que realmente es.

Al repetir la selección en cada fold, se deja de decir "estas son las mejores features, punto" y se pasa a decir algo más fuerte: "estas features siguen siendo elegidas aunque cambien los sujetos que el modelo ve durante el entrenamiento". Eso es una prueba mucho más creíble de que esas features realmente sirven, y no que parecían buenas solo porque se las mira con toda la información disponible.

### Configuración del split por sujeto (sin leakage)

- Entrenar por Sujeto, no por Ensayo

También se decide armar los grupos de entrenamiento/prueba por sujeto completo, no mezclando ensayos sueltos al azar. Esto sale directo de algo que se vió antes: en los gráficos de PCA, t-SNE y UMAP, los datos se agrupaban más por quién era la persona que por qué mano estaba imaginando mover.

Si se dejara que ensayos del mismo sujeto queden repartidos entre entrenamiento y prueba, el modelo podría "hacer trampa" de otra forma: en vez de aprender el patrón real de la imaginación motora, aprendería a reconocer la firma particular de esa persona (su forma de EEG, su ruido de fondo, etc.). Eso también infla artificialmente el resultado. Por eso se separa por sujeto entero: así el modelo se ve obligado a aprender algo que sirva para gente que nunca vio antes, que es justamente lo que se espera de un sistema BCI real.

In [2]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

X_all_features = df_features_wide.drop(columns=['subject_id', 'trial_idx', 'label', 'Clase_Nombre'])
y_all_features = df_features_wide['label']
groups = df_features_wide['subject_id']  # clave: el split respeta al sujeto, no al ensayo

# Limpieza de NaN y features ERD/ERS obsoletas
cols_erd = [c for c in X_all_features.columns if 'ERD' in c.upper() or 'ERS' in c.upper()]
if cols_erd:
    X_all_features = X_all_features.drop(columns=cols_erd)

nan_rows = X_all_features.isnull().any(axis=1)
if nan_rows.any():
    X_all_features = X_all_features[~nan_rows]
    y_all_features = y_all_features[~nan_rows]
    groups = groups[~nan_rows]

X_all_features = X_all_features.reset_index(drop=True)
y_all_features = y_all_features.reset_index(drop=True)
groups = groups.reset_index(drop=True)

k_all = 20


cv_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

print(f"X_all_features: {X_all_features.shape} | k_all = {k_all} | Sujetos: {groups.nunique()}")

X_all_features: (19032, 75) | k_all = 20 | Sujetos: 60


### Loop de selección dentro de cada fold (los 4 métodos)

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression

registros_seleccion = []  # una fila por (fold, método, feature seleccionada)

for fold_idx, (train_idx, test_idx) in enumerate(cv_splitter.split(X_all_features, y_all_features, groups)):
    X_train_fold = X_all_features.iloc[train_idx]
    y_train_fold = y_all_features.iloc[train_idx]

    # Verificación de no-leakage: ningún sujeto debe estar en ambos lados del split
    sujetos_train = set(groups.iloc[train_idx])
    sujetos_test = set(groups.iloc[test_idx])
    assert len(sujetos_train & sujetos_test) == 0, f"Fold {fold_idx}: leakage de sujetos detectado!"

    # --- Escalado: fit SOLO con train del fold ---
    scaler_fold = StandardScaler()
    X_train_scaled = scaler_fold.fit_transform(X_train_fold)

    # --- ANOVA ---
    sel_anova = SelectKBest(f_classif, k=k_all).fit(X_train_fold, y_train_fold)
    feats_anova = X_all_features.columns[sel_anova.get_support(indices=True)]

    # --- Mutual Information ---
    sel_mi = SelectKBest(mutual_info_classif, k=k_all).fit(X_train_fold.values, y_train_fold.values.astype(int))
    feats_mi = X_all_features.columns[sel_mi.get_support(indices=True)]

    # --- RFE ---
    est_rfe = LogisticRegression(max_iter=1000, random_state=42)
    sel_rfe = RFE(estimator=est_rfe, n_features_to_select=k_all, step=1).fit(X_train_scaled, y_train_fold)
    feats_rfe = X_all_features.columns[sel_rfe.get_support(indices=True)]

    # --- Lasso (C más chico que antes, para severidad comparable) ---
    lasso_fold = LogisticRegression(penalty='l1', solver='liblinear', C=0.03, random_state=42, max_iter=1000)
    lasso_fold.fit(X_train_scaled, y_train_fold)
    feats_lasso = X_all_features.columns[np.where(lasso_fold.coef_[0] != 0)[0]]

    for metodo, feats in [('ANOVA', feats_anova), ('MI', feats_mi), ('RFE', feats_rfe), ('Lasso', feats_lasso)]:
        for f in feats:
            registros_seleccion.append({'fold': fold_idx, 'metodo': metodo, 'feature': f})

    print(f"✔ Fold {fold_idx}: train={len(train_idx)} trials ({len(sujetos_train)} sujetos) | "
          f"ANOVA={len(feats_anova)}, MI={len(feats_mi)}, RFE={len(feats_rfe)}, Lasso={len(feats_lasso)}")

df_seleccion_folds = pd.DataFrame(registros_seleccion)

✔ Fold 0: train=15192 trials (48 sujetos) | ANOVA=20, MI=20, RFE=20, Lasso=39
✔ Fold 1: train=15192 trials (48 sujetos) | ANOVA=20, MI=20, RFE=20, Lasso=41
✔ Fold 2: train=15272 trials (48 sujetos) | ANOVA=20, MI=20, RFE=20, Lasso=41
✔ Fold 3: train=15272 trials (48 sujetos) | ANOVA=20, MI=20, RFE=20, Lasso=40
✔ Fold 4: train=15200 trials (48 sujetos) | ANOVA=20, MI=20, RFE=20, Lasso=40


### Agregación: estabilidad de cada feature a través de los folds

In [4]:
n_folds = cv_splitter.get_n_splits()

# Cuántos folds seleccionaron cada feature, por método
tabla_estabilidad = df_seleccion_folds.groupby(['feature', 'metodo']).size().unstack(fill_value=0)
tabla_estabilidad = tabla_estabilidad.reindex(columns=['ANOVA', 'MI', 'RFE', 'Lasso'], fill_value=0)

# Normalizado a "% de folds en que fue seleccionada" por método
tabla_estabilidad_pct = (tabla_estabilidad / n_folds * 100).round(0).astype(int)

# Consenso final: promedio de estabilidad entre los 4 métodos
tabla_estabilidad_pct['Consenso_Promedio_%'] = tabla_estabilidad_pct.mean(axis=1).round(1)
tabla_estabilidad_pct = tabla_estabilidad_pct.sort_values('Consenso_Promedio_%', ascending=False)

print(f"Tabla de estabilidad de features a través de {n_folds} folds (Group K-Fold por sujeto):")
display(tabla_estabilidad_pct.head(20))

Tabla de estabilidad de features a través de 5 folds (Group K-Fold por sujeto):


metodo,ANOVA,MI,RFE,Lasso,Consenso_Promedio_%
feature,,,,,
power_band_2_C3,100,80,60,100,85.0
AR_coef10_C4,100,20,100,100,80.0
potencia_total_C4,100,20,80,100,75.0
katz_fractal_C4,100,100,0,100,75.0
potencia_total_C3,80,40,80,100,75.0
power_band_1_C3,100,100,0,100,75.0
entropia_perm_C3,100,100,0,100,75.0
AR_coef10_C3,80,60,100,40,70.0
power_band_2_C4,100,20,60,100,70.0


A partir de la tabla de estabilidad, se observa que power_band_2_C3 es la feature más consistente entre los cuatro métodos (85% de consenso promedio): tanto ANOVA como Lasso la seleccionan en el 100% de los folds, MI en el 80%, y aunque RFE desciende a 60% es esperable debiado a su enfoque multivariado, distinto al de los otros tres métodos;sin embargo, la feature se mantiene relevante en la mayoría de las particiones.

En base a este resultado, se fija un criterio objetivo y explícito de corte para definir el conjunto final de features a utilizar en el entrenamiento de los modelos: se conservan únicamente aquellas variables cuyo Consenso_Promedio_% sea igual o superior al 65%, quedando así un subconjunto de features respaldadas por una mayoría consistente de los métodos de selección, en lugar de depender del resultado de uno solo.

### Guardado de checkpoint: Resultados de selección de features

In [5]:
import json

DIR_SALIDA_SELECCION = os.path.join(PATH_REPO, "data_procesada")
os.makedirs(DIR_SALIDA_SELECCION, exist_ok=True)

# 1. Detalle crudo: qué feature fue elegida por qué método, en qué fold
ruta_seleccion_folds = os.path.join(DIR_SALIDA_SELECCION, "seleccion_features_por_fold.parquet")
df_seleccion_folds.to_parquet(ruta_seleccion_folds, index=False)
print(f"Guardado: {ruta_seleccion_folds}")

# 2. Tabla resumen: % de estabilidad de cada feature por método + consenso
ruta_estabilidad = os.path.join(DIR_SALIDA_SELECCION, "tabla_estabilidad_features.csv")
tabla_estabilidad_pct.to_csv(ruta_estabilidad)
print(f"Guardado: {ruta_estabilidad}")

# 3. Features finales seleccionadas (corte por umbral de consenso)
UMBRAL_CONSENSO = 65  # % mínimo de consenso promedio para incluir la feature
features_finales = tabla_estabilidad_pct[tabla_estabilidad_pct['Consenso_Promedio_%'] >= UMBRAL_CONSENSO].index.tolist()

print(f"\nFeatures seleccionadas con consenso ≥ {UMBRAL_CONSENSO}%: {len(features_finales)}")
print(features_finales)

ruta_features_finales = os.path.join(DIR_SALIDA_SELECCION, "features_finales_seleccionadas.json")
with open(ruta_features_finales, 'w') as f:
    json.dump({
        'umbral_consenso': UMBRAL_CONSENSO,
        'n_features': len(features_finales),
        'features': features_finales
    }, f, indent=2)
print(f"Guardado: {ruta_features_finales}")



Guardado: /content/drive/MyDrive/BCI-MENTORIA-UNC/data_procesada/seleccion_features_por_fold.parquet
Guardado: /content/drive/MyDrive/BCI-MENTORIA-UNC/data_procesada/tabla_estabilidad_features.csv

Features seleccionadas con consenso ≥ 65%: 11
['power_band_2_C3', 'AR_coef10_C4', 'potencia_total_C4', 'katz_fractal_C4', 'potencia_total_C3', 'power_band_1_C3', 'entropia_perm_C3', 'AR_coef10_C3', 'power_band_2_C4', 'entropia_perm_C4', 'power_band_3_C4']
Guardado: /content/drive/MyDrive/BCI-MENTORIA-UNC/data_procesada/features_finales_seleccionadas.json
